In [3]:
pip install xgboost

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

print("Loading and preparing data...")
df = pd.read_csv("data/News_Final.csv")

# Clean tracking and drop rows with missing text so TF-IDF doesn't crash
df = df[df['Facebook'] >= 0]
df = df.dropna(subset=['Title', 'Headline'])

# Define target
median_fb = df['Facebook'].median()
df['Popular'] = (df['Facebook'] >= median_fb).astype(int)

# Feature Engineering
df['PublishDate'] = pd.to_datetime(df['PublishDate'])
df['Hour'] = df['PublishDate'].dt.hour
df['DayOfWeek'] = df['PublishDate'].dt.dayofweek
top_sources = df['Source'].value_counts().nlargest(20).index
df['Source_Clean'] = df['Source'].apply(lambda x: x if x in top_sources else 'Other')

# Feature Sets
numeric_features = ['SentimentTitle', 'SentimentHeadline', 'Hour', 'DayOfWeek']
categorical_features = ['Topic', 'Source_Clean']
text_feature_1 = 'Title'
text_feature_2 = 'Headline'

X = df[numeric_features + categorical_features + [text_feature_1, text_feature_2]]
y = df['Popular']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Building complex preprocessor with TF-IDF...")
# We use max_features to prevent memory crashes and severe overfitting
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
        ('title_tfidf', TfidfVectorizer(max_features=300, stop_words='english'), text_feature_1),
        ('head_tfidf', TfidfVectorizer(max_features=300, stop_words='english'), text_feature_2)
    ])

print("Defining base models and Stacking Classifier...")
# Base learners
rf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
xgb = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1, eval_metric='logloss')

# Meta learner
meta_model = LogisticRegression()

# Stacking Classifier
stack = StackingClassifier(
    estimators=[('rf', rf), ('xgb', xgb)],
    final_estimator=meta_model,
    cv=3,
    n_jobs=-1
)

final_pipeline = Pipeline([
    ('prep', preprocessor),
    ('clf', stack)
])

print("Training the Stacking Ensemble (This will take a few minutes)...")
final_pipeline.fit(X_train, y_train)

print("Evaluating...")
preds = final_pipeline.predict(X_test)

print("\n--- Advanced Architecture Results ---")
print(f"Accuracy:  {accuracy_score(y_test, preds):.4f}")
print(f"Precision: {precision_score(y_test, preds):.4f}")
print(f"Recall:    {recall_score(y_test, preds):.4f}")
print(f"F1 Score:  {f1_score(y_test, preds):.4f}")

Loading and preparing data...
Building complex preprocessor with TF-IDF...
Defining base models and Stacking Classifier...
Training the Stacking Ensemble (This will take a few minutes)...
Evaluating...

--- Advanced Architecture Results ---
Accuracy:  0.7019
Precision: 0.7668
Recall:    0.5837
F1 Score:  0.6628
